In [ ]:
import os
import cv2
import mysql.connector
import smtplib
from datetime import datetime
import face_recognition
from inference import get_model
import supervision as sv

DB_CONFIG = {
    "user": "root",
    "password": "",
    "host": "127.0.0.1",
    "database": "criminaldetection"
}

LOCATION = "CBS"
DETECTED_BY = "CCTV_CAM_01"
TOLERANCE = 0.6

KNOWN_DIRS = [
    r"missingpeopleimg",
    r"criminalpeopleimg"
]

EMAIL_FROM = "demotesting2022@yahoo.com"
EMAIL_TO = "samruddhirajole74@gmail.com"
EMAIL_USERNAME = "testmodule2018@gmail.com"
EMAIL_PASSWORD = "vrjb xcle vgjm pjmo"  # App password

weapon_model = get_model(
    model_id="weapon-detection-f1lih/1",
    api_key="vnPatUF48hUHjvTmM1vs"
)

box_annotator = sv.BoxAnnotator(thickness=2)
label_annotator = sv.LabelAnnotator(text_thickness=2, text_scale=0.6)

def get_db_connection():
    return mysql.connector.connect(**DB_CONFIG)

def insert_weapon_detection(name, wtype, confidence):
    conn = get_db_connection()
    cursor = conn.cursor()

    now = datetime.now()
    sql = """
        INSERT INTO detected_weapons
        (weapon_name, weapon_type, confidence, location, detected_by, detected_date, detected_time)
        VALUES (%s,%s,%s,%s,%s,%s,%s)
    """
    values = (
        name,
        wtype,
        float(confidence),
        LOCATION,
        DETECTED_BY,
        now.date(),
        now.time()
    )

    cursor.execute(sql, values)
    conn.commit()
    cursor.close()
    conn.close()

def send_email_alert(message):
    subject = "ALERT: Detection Event"
    msg = f"From: {EMAIL_FROM}\nTo: {EMAIL_TO}\nSubject: {subject}\n\n{message}"

    try:
        server = smtplib.SMTP("smtp.gmail.com", 587)
        server.starttls()
        server.login(EMAIL_USERNAME, EMAIL_PASSWORD)
        server.sendmail(EMAIL_FROM, EMAIL_TO, msg)
        server.quit()
    except Exception as e:
        print("Email failed:", e)

known_faces = []
known_names = []

for base_dir in KNOWN_DIRS:
    for person in os.listdir(base_dir):
        person_dir = os.path.join(base_dir, person)
        if not os.path.isdir(person_dir):
            continue

        for img_file in os.listdir(person_dir):
            if img_file.startswith("."):
                continue

            img_path = os.path.join(person_dir, img_file)
            image = face_recognition.load_image_file(img_path)
            enc = face_recognition.face_encodings(image)

            if enc:
                known_faces.append(enc[0])
                known_names.append(person)

print("[INFO] Faces loaded:", len(known_names))

cap = cv2.VideoCapture(0)
last_logged = {"face": None, "weapon": None}

while True:
    ret, frame = cap.read()
    if not ret:
        break

    face_locations = face_recognition.face_locations(frame, model="cnn")
    face_encodings = face_recognition.face_encodings(frame, face_locations)

    for enc, loc in zip(face_encodings, face_locations):
        matches = face_recognition.compare_faces(known_faces, enc, TOLERANCE)
        name = "Unknown"

        if True in matches:
            name = known_names[matches.index(True)]

            if last_logged["face"] != name:
                msg = f"{name} detected at {LOCATION} on {datetime.now()}"
                send_email_alert(msg)
                last_logged["face"] = name

        cv2.rectangle(frame, (loc[3], loc[0]), (loc[1], loc[2]), (0,255,0), 2)
        cv2.putText(frame, name, (loc[3], loc[2]+20),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0,255,0), 2)

    results = weapon_model.infer(frame)[0]
    detections = sv.Detections.from_inference(results)

    if len(detections) > 0:
        for cls, conf in zip(detections.class_name, detections.confidence):
            if last_logged["weapon"] != cls:
                insert_weapon_detection(
                    name=cls,
                    wtype="Firearm / Sharp Object",
                    confidence=conf
                )
                send_email_alert(f"Weapon Detected: {cls} ({conf:.2f}) at {LOCATION}")
                last_logged["weapon"] = cls

    frame = box_annotator.annotate(frame, detections)
    frame = label_annotator.annotate(frame, detections)

    cv2.imshow("Integrated Surveillance System", frame)

    if cv2.waitKey(1) & 0xFF == ord("q"):
        break

cap.release()
cv2.destroyAllWindows()
print("[INFO] System shutdown cleanly")
